TEXT MINING FINAL PROJECT


1. We import libraries and read the csv

In [ ]:
import os
%pip install pandas
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from pprint import pprint

from time import time

from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve, average_precision_score

# path to where you saved the training dataset that I provided
readin = ''

spitout= ''


#this is loading training dataset
filename ="reviews_translated.csv"
# loading the training data
data_train = pd.read_csv(os.path.join(readin, filename), sep=',', encoding='utf-8')
data_train.info()

In [ ]:
comments = data_train['Reviews_Translation']

comments.head()

In [ ]:
!pip install deep-translator tqdm

1. Preprocessing the data using Hannes functions

In [ ]:
import os
import re
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer, PorterStemmer
from nltk.corpus import stopwords
import spacy
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm

# Function to download NLTK resources
def download_nltk_resources():
    required_resources = ['wordnet', 'stopwords', 'punkt']
    for resource in required_resources:
        try:
            nltk.data.find(f'tokenizers/{resource}' if resource == 'punkt' else f'corpora/{resource}')
        except LookupError:
            nltk.download(resource)

download_nltk_resources()

# Load spaCy model; if missing, install it in the current Jupyter kernel environment.
try:
    sp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download as spacy_download

    print("Installing spaCy model 'en_core_web_sm' for this environment...")
    spacy_download('en_core_web_sm')
    sp = spacy.load('en_core_web_sm')

# Enable tqdm for pandas
tqdm.pandas()

# Initialize stemmers and lemmatizer
porter = SnowballStemmer("english")
lmtzr = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

In [ ]:
#General functions for the preprocessing
"""
This module provides helper functions for text preprocessing.
Each function applies punctuation removal and stopword removal, and then one of three options:
    0: Lowercasing only.
    1: Lowercasing plus stemming.
    2: Lemmatizing (using spaCy; original casing is preserved).

The functions return a string of tokens separated by spaces.
"""

def preprocess_lower(text):
    """
    Preprocess text by:
       - Converting to lowercase.
       - Removing punctuation.
       - Tokenizing.
       - Removing stopwords.

    Returns:
        str: A string of filtered tokens separated by spaces.
    """
    text_lower = text.lower()
    text_no_punct = re.sub(r'[^\w\s]', '', text_lower)
    tokens = word_tokenize(text_no_punct)
    filtered_tokens = [token for token in tokens if token not in STOP_WORDS]
    return " ".join(filtered_tokens)

def preprocess_stem(text):
    """
    Preprocess text by performing all steps in preprocess_lower() and then applying stemming.

    Returns:
        str: A string of stemmed tokens separated by spaces.
    """
    tokens = preprocess_lower(text).split()
    ps = PorterStemmer()
    stemmed_tokens = [ps.stem(token) for token in tokens]
    return " ".join(stemmed_tokens)

def preprocess_lemma(text):
    """
    Preprocess text by:
       - Removing punctuation and stopwords using spaCy's token attributes.
       - Lemmatizing the text.
       - (Note: This function does NOT lowercase the text.)

    Returns:
        str: A string of lemmatized tokens separated by spaces.
    """
    doc = sp(text)
    lemmatized_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.lemma_.strip() != '']
    return " ".join(lemmatized_tokens)

def tokenize(text, mode=0):
    """
    General tokenize function. Always applies punctuation and stopword removal and then:

      mode = 0: Applies lowercasing.
      mode = 1: Applies lowercasing and stemming.
      mode = 2: Applies lemmatization (without lowercasing the original text).

    Args:
        text (str): The input text to be processed.
        mode (int): Processing mode (0 for lowercasing; 1 for stemming; 2 for lemmatizing).

    Returns:
        str: A string of processed tokens separated by spaces.

    Raises:
        ValueError: If an invalid mode is provided.
    """
    if mode == 0:
        return preprocess_lower(text)
    elif mode == 1:
        return preprocess_stem(text)
    elif mode == 2:
        return preprocess_lemma(text)
    else:
        raise ValueError("Invalid mode. Please use 0 for lowercasing, 1 for stemming, or 2 for lemmatizing.")


In [ ]:
#Step 1

#DONT TOUCH
#1. Remove punctuation and digits and remove stopwords, and lemmatizing
mod = 2

# Pre-process the text column with progress tracking
try:
    preprocessed_comments = comments.astype(str).progress_apply(lambda row: tokenize(row, mod))
    print("Done processing text.")
except Exception as e:
    print(f"Error processing text column: {e}")
    raise RuntimeError(f"Error processing text column: {e}") from e

print(preprocessed_comments.head())

In [ ]:
preprocessed_comments.to_csv('preprocessed_comments.csv', index=False)
print("La columna 'preprocessed_comments' se ha guardado en 'preprocessed_comments.csv'")

HERE WE START IMPLEMENTING THE FOLLOWING METHODOLOGY:
1- Descriptive statistics will be computed, including:
    - Number of reviews
    - Average review length
    - Distribution of review lengths
    - Most frequent words in the dataset
  These summaries provide an overview of the dataset.


In [ ]:
#DATA EXPLORATION
#Length of our dataset
print("Number of reviews:", len(comments))

#Check and remove missing values
print("Number of missing values:", comments.isnull().sum())
comments = comments.dropna()
# Para quitar una fila específica por su índice (por ejemplo, el índice 68051) de un objeto Series de pandas,
# debes usar el método `.drop()` directamente en el Series, pasándole el índice que deseas eliminar.
# El código anterior `comments[68051].drop()` falló porque `comments[68051]` devuelve el *contenido* de la celda (una cadena),
# y las cadenas no tienen un método `.drop()`.
comments = comments.drop(68051)

#Length of each review
review_length = comments.apply(lambda x: len(x.split()))
review_length.describe()

In [ ]:
review_length.sort_values(ascending=False).head(10)

In [ ]:
comments[71415]

In [ ]:
import matplotlib.pyplot as plt

plt.hist(review_length, bins=50)
plt.xlabel("Number of words in review")
plt.ylabel("Frequency")
plt.title("Distribution of Airbnb Review Lengths")

plt.show()

TF_IDF Implementation:
Term Frequency–Inverse Document Frequency (TF-IDF) will be used to identify words that are most important for distinguishing positive and negative reviews.
Separate TF-IDF scores will be analyzed for positive and negative reviews.
This helps identify the most characteristic words in each group.




In [ ]:
#TF_IDF

Topic Modeling (LDA)
Latent Dirichlet Allocation (LDA) will be used to identify latent topics in the reviews.
Each topic will be interpreted based on its most important words.
This helps uncover common themes such as cleanliness, location, or communication.


In [ ]:
#LDA

A machine learning classifier will be trained to predict whether a review is positive or negative.
Possible models include Logistic Regression and Naive Bayes.
Model performance will be evaluated using accuracy and confusion matrices.


Supervised Classification
A machine learning classifier will be trained to predict whether a review is positive or negative.
Possible models include Logistic Regression and Naive Bayes.
Model performance will be evaluated using accuracy and confusion matrices.


In [ ]:
#Supervised Classification